In Class April 7th

In [1]:
!pip install sweetviz

zsh:1: command not found: pip


In [3]:
#libs
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Matplotlib is building the font cache; this may take a moment.


In [4]:
insurance = pd.read_csv('../Data/insurance.csv')
insurance.head()

report = sv.analyze(insurance)
report.show_html('report.html')

[Summarizing dataframe]                      |          | [  0%]   00:00 -> (? left)

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [5]:
#encode categorical variables
insurance_encoded = pd.get_dummies(insurance, columns=['sex', 'smoker', 'region'], drop_first=True).astype(int)
insurance_encoded.head()



,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


In [6]:
#prepare data
X = insurance_encoded.drop('charges', axis=1)
y = insurance_encoded['charges']

#split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#scale data
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#baseline linear regression
baseline_model_lr = LinearRegression()
baseline_model_lr.fit(X_train, y_train)
y_pred_lr = baseline_model_lr.predict(X_test)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

#random forest regression
baseline_model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
baseline_model_rf.fit(X_train, y_train)
y_pred_rf = baseline_model_rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

#evaluate models
print(f"Linear Regression:")
print(f"MSE: {mse_lr:.2f}")
print(f"R2: {r2_lr:.2f}")
print(f"Random Forest Regression:")
print(f"MSE: {mse_rf:.2f}")
print(f"R2: {r2_rf:.2f}")


Linear Regression:
MSE: 33566439.74
R2: 0.78
Random Forest Regression:
MSE: 22118860.12
R2: 0.86


In [7]:
#CV with baseline models

cv_lr = cross_val_score(baseline_model_lr, X_train, y_train, cv=5, scoring='r2')
cv_rf = cross_val_score(baseline_model_rf, X_train, y_train, cv=5, scoring='r2')

cv_lr_r2 = cv_lr.mean()
cv_rf_r2 = cv_rf.mean()

print(f"Linear Regression CV R2: {cv_lr_r2:.2f}")
print(f"Random Forest Regression CV R2: {cv_rf_r2:.2f}")



Linear Regression CV R2: 0.73
Random Forest Regression CV R2: 0.82


In [8]:
#grid search for random forest
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [1, 2, 5],
    'max_features': ['sqrt', 'auto']
}

grid_search = GridSearchCV(estimator=baseline_model_rf, param_grid=param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)

#evaluate grid search
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best R2 Score: {grid_search.best_score_:.2f}")

#evaluate best model
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:490: FitFailedWarning: 
180 fits failed out of a total of 270.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
45 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/base.py", line 1329, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~

Best Parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 300}
Best R2 Score: 0.83


In [9]:
#examine top 20 ranked features
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False).head(20)

print(feature_importances)

            Feature  Importance
4        smoker_yes    0.604611
0               age    0.171836
1               bmi    0.148585
2          children    0.035867
3          sex_male    0.013502
6  region_southeast    0.009610
7  region_southwest    0.008111
5  region_northwest    0.007878


In [10]:
# Extract top 20 models by tuning
top_models = pd.DataFrame(grid_search.cv_results_).sort_values(by='mean_test_score', ascending=False).head(20)

# Extract parameter values and scores
top_models['param_values'] = top_models['params'].apply(lambda x: list(x.values()))
top_models = top_models[['param_values', 'mean_test_score']]


print(top_models)


            param_values  mean_test_score
23    [10, sqrt, 2, 300]         0.827364
22    [10, sqrt, 2, 200]         0.825674
5   [None, sqrt, 2, 300]         0.824825
41    [20, sqrt, 2, 300]         0.824800
26    [10, sqrt, 5, 300]         0.824451
24    [10, sqrt, 5, 100]         0.823630
21    [10, sqrt, 2, 100]         0.823572
40    [20, sqrt, 2, 200]         0.823311
4   [None, sqrt, 2, 200]         0.823297
25    [10, sqrt, 5, 200]         0.823189
44    [20, sqrt, 5, 300]         0.822761
8   [None, sqrt, 5, 300]         0.822761
39    [20, sqrt, 2, 100]         0.820900
3   [None, sqrt, 2, 100]         0.820803
7   [None, sqrt, 5, 200]         0.820763
43    [20, sqrt, 5, 200]         0.820763
6   [None, sqrt, 5, 100]         0.819870
42    [20, sqrt, 5, 100]         0.819870
0   [None, sqrt, 1, 100]              NaN
1   [None, sqrt, 1, 200]              NaN


I would choose the top model in the list with parameters 10, sqrt, 2 and 300. This is becasue in the current context it provdies a reasonably higher cross validated test score then the other options. If computing was a restricitve aspect of the project I may go down to the 6th model that has n estimators as 100 because this model will be much more friendly to tune and train. 

In [11]:
best_model = grid_search.best_estimator_

# Predictions for training and test sets
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

# Calculate R2 scores for training and test sets
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Print the comparison
print(f"Training R2 Score: {train_r2:.2f}")
print(f"Test R2 Score: {test_r2:.2f}")

Training R2 Score: 0.95
Test R2 Score: 0.85


In this context there may be slight overfitting as the training R2 score is higher than the test R2 score, but the difference is not too much. The model seems to generalize reasonably well to unseen data, showing that it captures the underlying patterns without being overly complex.